In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
# Dataset
data = [
"machine learning models learn patterns from data",
"sequence models process data step by step",
"recurrent neural networks are designed for sequential tasks",
"rnn models maintain hidden states across time steps",
"long short term memory networks solve long dependency problems",
"lstm uses gates to control information flow",
"gru models simplify the lstm architecture",
"sequence prediction is useful in many applications",
"language modeling predicts the next word in a sentence",
"speech recognition processes audio sequences",
"time series forecasting predicts future values",
"music generation creates new melodies",
"generative models learn probability distributions",
"they generate new samples similar to training data",
"sequence generation is widely used in artificial intelligence",
"deep learning improves sequence modeling performance"
]

In [ ]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)

total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []

for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

# Padding
max_seq_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

# Split input and label
X = input_sequences[:,:-1]
y = input_sequences[:,-1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Build LSTM model
model = Sequential()

model.add(Embedding(total_words, 64, input_length=max_seq_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=200, verbose=1)

# Generate text
def generate_text(seed_text, next_words):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list), axis=-1)

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break

    return seed_text


print(generate_text("machine learning", 5))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.0105 - loss: 4.4675   
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0737 - loss: 4.4590
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0526 - loss: 4.4514
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0526 - loss: 4.4424    
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0526 - loss: 4.4321
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0526 - loss: 4.4150
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0526 - loss: 4.3870
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0526 - loss: 4.3504
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0526 - loss: 4.2844
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0526 - loss: 4.2460
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0526 - loss: 4.2374
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.05

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Dense, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras import Input
import numpy as np

# Dataset
dataset = [
"machine learning models learn patterns from data",
"sequence models process data step by step",
"recurrent neural networks are designed for sequential tasks",
"rnn models maintain hidden states across time steps",
"long short term memory networks solve long dependency problems",
"lstm uses gates to control information flow",
"gru models simplify the lstm architecture",
"sequence prediction is useful in many applications",
"language modeling predicts the next word in a sentence",
"speech recognition processes audio sequences",
"time series forecasting predicts future values",
"music generation creates new melodies",
"generative models learn probability distributions",
"they generate new samples similar to training data",
"sequence generation is widely used in artificial intelligence",
"deep learning improves sequence modeling performance"
]

# Vectorization
vectorize_layer = TextVectorization(max_tokens=1000, output_mode='int', output_sequence_length=10)
vectorize_layer.adapt(dataset)

X = vectorize_layer(dataset)
y = np.roll(X, -1, axis=1)

# Transformer Block
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# Model
inputs = Input(shape=(10,))
x = Embedding(1000, 32)(inputs)
x = TransformerBlock(32, 2, 32)(x)
x = GlobalAveragePooling1D()(x)
outputs = Dense(1000, activation="softmax")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

# Train
model.fit(X, y[:,0], epochs=40, verbose=0)

# Generate
def generate_transformer(seed_text):
    seq = vectorize_layer([seed_text])
    pred = model.predict(seq, verbose=0)
    index = np.argmax(pred)

    vocab = vectorize_layer.get_vocabulary()
    if index < len(vocab):
        return seed_text + " " + vocab[index]
    return seed_text

# Output
print("\n===== TRANSFORMER GENERATED OUTPUT =====\n")

seeds = ["machine learning", "sequence prediction", "deep learning", "language modeling"]

for seed in seeds:
    print("Input :", seed)
    print("Output:", generate_transformer(seed))
    print("-"*50)


===== TRANSFORMER GENERATED OUTPUT =====

Input : machine learning
Output: machine learning models
--------------------------------------------------
Input : sequence prediction
Output: sequence prediction models
--------------------------------------------------
Input : deep learning
Output: deep learning models
--------------------------------------------------
Input : language modeling
Output: language modeling models
--------------------------------------------------
